# Pipeline Công nợ & Trích xuất Hóa đơn — Chạy trên Kaggle (GPU)

4 **file** `.py` đã test kỹ ở local, mỗi file = 1 nhóm cell trên Kaggle (ghi file bằng
`%%writefile` rồi chạy thử ngay). Giữ nguyên kiến trúc import gốc, không viết lại logic.

Lưu ý kỹ thuật: `%%writefile` bắt buộc là dòng đầu tiên và DUY NHẤT lệnh trong 1 cell — nên
phần cài đặt package / chạy thử phải tách thành cell riêng ngay sau, không gộp chung được.

**Chạy tuần tự từ trên xuống, không đảo thứ tự.** Trước khi chạy: Kaggle → Settings (phải) →
Accelerator → chọn GPU T4 x2 (hoặc P100).

## File 1 — `ap_analysis.py` (nguồn logic gốc)

Schema chuẩn + công thức DPO/aging/forecast. Không cần cài thêm package (pandas/numpy có sẵn
trên Kaggle).

In [1]:
%%writefile ap_analysis.py
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# =============================================================
# 1. SINH DỮ LIỆU MẪU (mô phỏng hệ thống kế toán thực tế hơn)
# =============================================================
np.random.seed(42)
n = 50

vendors = [
    'Công ty CP Thép Hòa An',
    'Nhà cung cấp Nhựa Tân Phát',
    'Bao bì Toàn Cầu',
    'Hóa chất Việt Đức',
    'Điện máy Công nghiệp Phúc Hưng'
]
terms_list = ['Net 30', 'Net 60', '2/10 Net 30']
start_date = datetime(2026, 1, 1)

def generate_invoices(n=50):
    rows = []
    for i in range(1, n + 1):
        vendor = np.random.choice(vendors)
        term = np.random.choice(terms_list, p=[0.4, 0.3, 0.3])
        inv_date = start_date + timedelta(days=int(np.random.randint(0, 90)))
        term_days = 60 if term == 'Net 60' else 30
        due_date = inv_date + timedelta(days=term_days)
        amount = float(np.random.randint(20, 350) * 1_000_000)

        # Ghi chú (2): số hóa đơn gốc/ký hiệu để đối chiếu pháp lý, tách khỏi mã nội bộ
        symbol = f"1C26T{np.random.randint(1,4)}"
        legal_no = f"{np.random.randint(1, 99999):07d}"

        # Ghi chú (3): hóa đơn điều chỉnh/giảm giá (credit note) ~15% số hóa đơn
        credit_note = round(amount * 0.05, 0) if np.random.rand() < 0.15 else 0.0
        net_amount = amount - credit_note

        # Ghi chú (1): thanh toán từng đợt thay vì nhị phân trả hết/chưa trả
        n_installments = np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])
        remaining = net_amount
        payments = []
        cursor_date = inv_date
        for k in range(n_installments):
            if np.random.rand() < 0.25 and k == n_installments - 1:
                break  # đợt cuối chưa thanh toán -> hóa đơn còn dư nợ
            pct = 1.0 if n_installments == 1 else np.random.uniform(0.3, 0.6)
            pay_amt = round(min(remaining, net_amount * pct), 0)
            if pay_amt <= 0:
                continue
            gap = max(3, int(np.random.normal(loc=term_days / n_installments, scale=10)))
            cursor_date = cursor_date + timedelta(days=gap)
            dispute = np.random.rand() < 0.08          # (4) tranh chấp -> mất quyền chiết khấu
            wrong_channel = np.random.rand() < 0.05     # (4) sai kênh thỏa thuận -> mất quyền chiết khấu
            payments.append({
                'pay_date': cursor_date,
                'pay_amount': pay_amt,
                'dispute': dispute,
                'wrong_channel': wrong_channel
            })
            remaining -= pay_amt

        rows.append({
            'Invoice_ID': f"INV-2026-{i:03d}",
            'Legal_Invoice_No': legal_no,
            'Invoice_Symbol': symbol,
            'Vendor_Name': vendor,
            'Invoice_Date': inv_date,
            'Payment_Terms': term,
            'Due_Date': due_date,
            'Gross_Amount': amount,
            'Credit_Note_Amount': credit_note,
            'Net_Amount': net_amount,
            'Payments': payments,   # list thay vì 1 giá trị đơn -> hỗ trợ trả nhiều đợt
        })
    return pd.DataFrame(rows)

df = generate_invoices(n)

# =============================================================
# 2. LÀM SẠCH DỮ LIỆU (Data Integrity)
# =============================================================

# (5) Ngày hóa đơn phải <= ngày đến hạn — hóa đơn nào vi phạm bị gắn cờ để loại khỏi báo cáo
df['Date_Anomaly'] = df['Invoice_Date'] > df['Due_Date']

# (5) Phát hiện trùng lặp: cùng NCC + cùng số tiền + hóa đơn cách nhau <=1 ngày
# (thường do quét OCR trùng file) -> gắn cờ, không tự xóa để kế toán xác nhận thủ công
df_sorted = df.sort_values(['Vendor_Name', 'Gross_Amount', 'Invoice_Date'])
df['Possible_Duplicate'] = df_sorted.duplicated(subset=['Vendor_Name', 'Gross_Amount'], keep=False) & \
    (df_sorted.groupby(['Vendor_Name', 'Gross_Amount'])['Invoice_Date']
     .transform(lambda s: (s.max() - s.min()).days <= 1))
df['Possible_Duplicate'] = df['Possible_Duplicate'].reindex(df.index, fill_value=False)

df_clean = df[~df['Date_Anomaly']].copy()

# =============================================================
# 3. PHÂN TÍCH POINT-IN-TIME theo mốc chốt sổ (không hardcode)
# =============================================================

REQUIRED_COLUMNS = [
    "Invoice_ID", "Vendor_Name", "Invoice_Date", "Payment_Terms",
    "Due_Date", "Net_Amount", "Payments",
]


def analyze(df_in, as_of_date=None):
    """Toàn bộ trạng thái công nợ phải được tính LẠI theo as_of_date,
    không dùng trạng thái 'hiện tại' của Payments (tránh Data Leakage)."""
    if df_in.empty:
        raise ValueError(
            "analyze() nhận vào DataFrame rỗng (0 hóa đơn) — không có gì để phân tích. "
            "Kiểm tra lại bước trích xuất/lọc trước đó: có thể thư mục ảnh trống, "
            "hoặc toàn bộ hóa đơn bị clean_extracted_record()/validate_extracted_data() loại bỏ."
        )
    missing = [c for c in REQUIRED_COLUMNS if c not in df_in.columns]
    if missing:
        raise ValueError(
            f"analyze() thiếu cột bắt buộc: {missing}. "
            f"DataFrame truyền vào có các cột: {list(df_in.columns)}. "
            "Kiểm tra lại pipeline trích xuất — schema đầu ra phải khớp REQUIRED_COLUMNS."
        )
    as_of_date = pd.to_datetime(as_of_date or datetime.now().date())
    out = df_in.copy()

    def paid_by(payments):
        return sum(p['pay_amount'] for p in payments if p['pay_date'] <= as_of_date)

    def last_pay_date(payments):
        past = [p['pay_date'] for p in payments if p['pay_date'] <= as_of_date]
        return max(past) if past else pd.NaT

    def discount_ok(payments, net_amount):
        # (4) Điều kiện chiết khấu thật: trả đủ trong <=10 ngày, không tranh chấp, đúng kênh
        past = [p for p in payments if p['pay_date'] <= as_of_date]
        if not past:
            return False
        total_paid = sum(p['pay_amount'] for p in past)
        if total_paid < net_amount * 0.999:
            return False
        if any(p['dispute'] or p['wrong_channel'] for p in past):
            return False
        return max(p['pay_date'] for p in past) <= out.loc[0, 'Invoice_Date']  # placeholder, overwritten below

    out['Paid_To_Date'] = out['Payments'].apply(paid_by)
    out['Outstanding'] = (out['Net_Amount'] - out['Paid_To_Date']).round(0)
    out['Is_Fully_Paid'] = out['Outstanding'] <= 0
    out['Last_Payment_Date'] = out['Payments'].apply(last_pay_date)
    out['Days_To_Pay'] = (out['Last_Payment_Date'] - out['Invoice_Date']).dt.days

    # điều kiện chiết khấu tính đúng (không dùng hàm placeholder ở trên)
    def eligible_discount(row):
        if row['Payment_Terms'] != '2/10 Net 30' or not row['Is_Fully_Paid']:
            return False
        past = [p for p in row['Payments'] if p['pay_date'] <= as_of_date]
        if any(p['dispute'] or p['wrong_channel'] for p in past):
            return False
        return (row['Last_Payment_Date'] - row['Invoice_Date']).days <= 10
    out['Discount_Eligible'] = out.apply(eligible_discount, axis=1)
    out['Missed_Discount_VND'] = np.where(
        (out['Payment_Terms'] == '2/10 Net 30') & out['Is_Fully_Paid'] & ~out['Discount_Eligible'],
        out['Net_Amount'] * 0.02, 0.0)

    # (6) DPO theo trọng số giá trị tiền (dollar-weighted), không phải trung bình cộng số ngày
    paid_rows = out[out['Is_Fully_Paid'] & out['Days_To_Pay'].notna()]
    dpo_dollar_weighted = (
        (paid_rows['Days_To_Pay'] * paid_rows['Net_Amount']).sum() / paid_rows['Net_Amount'].sum()
        if len(paid_rows) else np.nan
    )

    # Aging schedule cho phần dư nợ chưa trả hết tính đến as_of_date
    open_rows = out[~out['Is_Fully_Paid']].copy()
    open_rows['Overdue_Days'] = (as_of_date - open_rows['Due_Date']).dt.days

    def bucket(d):
        if d <= 0: return '1. Chưa đến hạn'
        if d <= 30: return '2. Quá hạn 1-30 ngày'
        if d <= 60: return '3. Quá hạn 31-60 ngày'
        return '4. Quá hạn >60 ngày (rủi ro cao)'
    open_rows['Aging_Bucket'] = open_rows['Overdue_Days'].apply(bucket)

    aging_report = open_rows.groupby('Aging_Bucket')['Outstanding'].agg(
        So_luong_HD='count', Tong_no_VND='sum').reset_index()
    if aging_report['Tong_no_VND'].sum() > 0:
        aging_report['Ty_le_%'] = (aging_report['Tong_no_VND'] / aging_report['Tong_no_VND'].sum() * 100).round(1)

    # (7) Mức độ tập trung nhà cung cấp trên phần dư nợ quá hạn
    overdue_rows = open_rows[open_rows['Overdue_Days'] > 0]
    vendor_concentration = overdue_rows.groupby('Vendor_Name')['Outstanding'].sum().sort_values(ascending=False)
    vendor_concentration_pct = (vendor_concentration / vendor_concentration.sum() * 100).round(1) \
        if vendor_concentration.sum() > 0 else vendor_concentration

    # (8) Dự báo dòng tiền ra: hóa đơn CHƯA quá hạn nhưng sẽ đến hạn trong 7/14/30 ngày tới
    upcoming = open_rows[open_rows['Overdue_Days'] <= 0]
    forecast = {}
    for horizon in (7, 14, 30):
        mask = upcoming['Due_Date'] <= as_of_date + timedelta(days=horizon)
        forecast[horizon] = upcoming.loc[mask, 'Outstanding'].sum()

    return {
        'as_of_date': as_of_date,
        'dpo_dollar_weighted': dpo_dollar_weighted,
        'missed_discount_total': out['Missed_Discount_VND'].sum(),
        'aging_report': aging_report,
        'vendor_concentration_pct': vendor_concentration_pct,
        'cash_outflow_forecast': forecast,
        'flags': {
            'date_anomalies': int(df['Date_Anomaly'].sum()),
            'possible_duplicates': int(df['Possible_Duplicate'].sum()),
        }
    }

# =============================================================
# 4. BÁO CÁO (as_of_date truyền tham số, không hardcode)
# =============================================================
# Chỉ chạy demo report khi gọi trực tiếp "python ap_analysis.py" — không chạy khi bị
# import bởi paddleocr_qwen_extraction.py / invoice_validation.py / streamlit_app.py,
# tránh in báo cáo dữ liệu GIẢ LẬP đè lên output thật của các file đó mỗi lần import.
if __name__ == "__main__":
    result = analyze(df_clean, as_of_date='2026-04-15')

    print("=" * 60)
    print(f"BÁO CÁO CÔNG NỢ & VỐN LƯU ĐỘNG (tính đến {result['as_of_date'].strftime('%d/%m/%Y')})")
    print("=" * 60)
    print(f"1. DPO (dollar-weighted): {result['dpo_dollar_weighted']:.1f} ngày")
    print(f"2. Chiết khấu sớm bị bỏ lỡ: {result['missed_discount_total']:,.0f} VNĐ")
    print(f"3. Cảnh báo dữ liệu: {result['flags']['date_anomalies']} ngày bất thường, "
          f"{result['flags']['possible_duplicates']} hóa đơn nghi trùng lặp\n")

    print("4. AGING SCHEDULE (phần dư nợ, không phải tổng hóa đơn):")
    for _, row in result['aging_report'].iterrows():
        print(f" - {row['Aging_Bucket']}: {row['So_luong_HD']} hóa đơn | {row['Tong_no_VND']:,.0f} VNĐ ({row.get('Ty_le_%', 0)}%)")

    print("\n5. TẬP TRUNG RỦI RO NHÀ CUNG CẤP (trên dư nợ quá hạn):")
    for vendor, pct in result['vendor_concentration_pct'].items():
        print(f" - {vendor}: {pct}%")

    print("\n6. DỰ BÁO DÒNG TIỀN RA (hóa đơn CHƯA quá hạn, sắp đến hạn):")
    for horizon, amt in result['cash_outflow_forecast'].items():
        print(f" - {horizon} ngày tới: {amt:,.0f} VNĐ")


Writing ap_analysis.py


In [2]:
# Chạy thử — demo báo cáo dữ liệu giả lập
!python ap_analysis.py

BÁO CÁO CÔNG NỢ & VỐN LƯU ĐỘNG (tính đến 15/04/2026)
1. DPO (dollar-weighted): 30.2 ngày
2. Chiết khấu sớm bị bỏ lỡ: 23,820,000 VNĐ
3. Cảnh báo dữ liệu: 0 ngày bất thường, 0 hóa đơn nghi trùng lặp

4. AGING SCHEDULE (phần dư nợ, không phải tổng hóa đơn):
 - 1. Chưa đến hạn: 13 hóa đơn | 1,767,871,250 VNĐ (51.6%)
 - 2. Quá hạn 1-30 ngày: 4 hóa đơn | 388,934,459 VNĐ (11.4%)
 - 3. Quá hạn 31-60 ngày: 11 hóa đơn | 1,026,582,819 VNĐ (30.0%)
 - 4. Quá hạn >60 ngày (rủi ro cao): 2 hóa đơn | 243,168,784 VNĐ (7.1%)

5. TẬP TRUNG RỦI RO NHÀ CUNG CẤP (trên dư nợ quá hạn):
 - Công ty CP Thép Hòa An: 29.3%
 - Hóa chất Việt Đức: 25.7%
 - Nhà cung cấp Nhựa Tân Phát: 19.1%
 - Điện máy Công nghiệp Phúc Hưng: 19.0%
 - Bao bì Toàn Cầu: 6.9%

6. DỰ BÁO DÒNG TIỀN RA (hóa đơn CHƯA quá hạn, sắp đến hạn):
 - 7 ngày tới: 165,517,416 VNĐ
 - 14 ngày tới: 1,271,021,250 VNĐ
 - 30 ngày tới: 1,539,871,250 VNĐ


## File 2 — `paddleocr_qwen_extraction.py` (OCR + Qwen)

**Trước khi chạy:** upload ảnh hóa đơn vào `/kaggle/working/invoices_scan/` — panel "Add Data"
bên phải, hoặc kéo thả trực tiếp vào file browser Kaggle.

GPU Kaggle (T4/P100) đủ VRAM cho cả PaddleOCR lẫn Qwen2.5-3B-Instruct — không cần
quant/bitsandbytes/AWQ như lúc chạy CPU/VRAM thấp ở local nữa.

In [3]:
# Cài PaddleOCR bằng CPU — KHÔNG dùng paddlepaddle-gpu.
# Lý do: paddlepaddle-gpu kéo theo các gói nvidia-*-cu12 riêng, phiên bản khác với
# nvidia-*-cu12 mà torch (Kaggle cài sẵn, cu128) đang cần -> cài đè lên nhau, vỡ torch
# (đã gặp thật: lỗi "undefined symbol: ncclCommShrink" rồi tiếp là hàng loạt cảnh báo
# "torch requires nvidia-X==... but you have nvidia-X==... incompatible").
# Qwen (dùng torch) vẫn chạy GPU bình thường vì không đụng vào paddlepaddle -> tách 2
# framework GPU ra là cách an toàn nhất, tránh xung đột nvidia-*-cu12 giữa chúng.
!pip install -q paddleocr
!pip install -q paddlepaddle==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!mkdir -p invoices_scan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 113.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
%%writefile paddleocr_qwen_extraction.py
"""
Pipeline trích xuất hóa đơn: PaddleOCR (layout/text) -> Qwen (structuring) -> DataFrame
tương thích schema đầu vào của analyze() trong ap_analysis.py.

Phiên bản khuyến nghị:
- PaddleOCR PP-OCRv4 (lang='vi' hoặc 'vietnamese' tùy version) — nhẹ, hỗ trợ tiếng Việt tốt,
  dùng PP-StructureV2 nếu cần giữ layout bảng (nhiều hóa đơn có bảng chi tiết dòng hàng).
  KHÔNG dùng PP-OCRv5 nếu máy không có GPU đời mới — v4 ổn định hơn cho CPU/Colab free tier.
- Qwen2.5-3B-Instruct (bản gốc, KHÔNG quant) để cấu trúc hóa văn bản OCR thành JSON.
  Đã thử AWQ/bitsandbytes cho bản 7B nhưng cả 2 đều gãy trên máy VRAM/RAM thấp:
  bitsandbytes cần load tạm bản fp16 đầy đủ (~15GB) qua RAM trước khi nén -> tràn RAM;
  AutoAWQ (thư viện quant AWQ) đã NGỪNG BẢO TRÌ chính thức, không còn tương thích các bản
  torch/transformers mới. Bản 3B ở fp16 chỉ cần ~6GB VRAM, vừa khít RTX 3050 Ti, không cần
  quant gì cả -> tránh toàn bộ nhóm lỗi trên. Đủ tốt cho tác vụ structuring JSON (không phải
  sinh văn bản dài/sáng tạo, không cần model lớn).
  Nếu cần đọc trực tiếp ảnh hóa đơn mất khung/nghiêng nặng, đổi sang Qwen2.5-VL-3B-Instruct
  (bỏ qua bước OCR text, đưa thẳng ảnh vào model).
  => Pipeline dưới đây dùng cấu hình: PaddleOCR trích text -> Qwen2.5-3B-Instruct structuring,
     phù hợp GPU 6-8GB VRAM như RTX 3050 Ti (laptop), không cần quant/bitsandbytes/AWQ.

Giữ đúng 4 bước Extract/Clean/Map/Export như doc-normalization pipeline đã dùng trước đó.
"""

import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from paddleocr import PaddleOCR
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Import lại schema/hàm phân tích đã có sẵn để pipeline này "kế thừa" trực tiếp
from ap_analysis import analyze, terms_list  # noqa: F401  (terms_list dùng để validate Payment_Terms)

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# =============================================================
# BƯỚC 1 — EXTRACT: OCR ảnh/PDF hóa đơn bằng PaddleOCR
# =============================================================

def build_ocr_engine(lang: str = "vi") -> PaddleOCR:
    """Khởi tạo PaddleOCR 1 lần, tái sử dụng cho nhiều file (tránh load lại model mỗi lần gọi).
    PaddleOCR 3.x (từ 04/2025) đổi API không tương thích ngược với 2.x:
    - use_angle_cls / show_log đã bị XÓA khỏi constructor -> dùng use_textline_orientation thay thế
    - Muốn tắt log thì cấu hình qua logging chuẩn của Python, không truyền tham số vào PaddleOCR nữa"""
    import logging
    logging.getLogger("ppocr").setLevel(logging.ERROR)
    return PaddleOCR(lang=lang, use_textline_orientation=True)


def ocr_extract_text(ocr_engine: PaddleOCR, image_path: str) -> str:
    """Chạy OCR trên 1 ảnh/scan hóa đơn, trả về text thô theo thứ tự đọc trên xuống dưới.
    Không tự parse field ở đây — nhiệm vụ của bước này CHỈ là lấy text, tránh lẫn logic.
    PaddleOCR 3.x đổi .ocr(cls=True) -> .predict(); output không còn dạng [box, (text, conf)]
    mà là list các dict có 'rec_texts' / 'rec_scores'."""
    result = ocr_engine.predict(image_path)
    lines = []
    for page in result:
        texts = page.get("rec_texts", [])
        scores = page.get("rec_scores", [])
        for text, score in zip(texts, scores):
            if score >= 0.5:  # bỏ text OCR quá nhiễu, độ tin cậy thấp
                lines.append(text)
    return "\n".join(lines)


# =============================================================
# BƯỚC 2 — MAP: dùng Qwen cấu trúc hóa text OCR thành JSON theo schema
# =============================================================

EXTRACTION_SCHEMA_PROMPT = """Bạn là công cụ trích xuất dữ liệu hóa đơn kế toán tiếng Việt.
Từ văn bản OCR dưới đây, trả về DUY NHẤT một JSON object (không thêm chữ nào khác) với các khóa:
- "legal_invoice_no": số hóa đơn gốc (chuỗi số, không lấy mã nội bộ)
- "invoice_symbol": ký hiệu mẫu số hóa đơn (vd "1C26TAA")
- "vendor_name": tên nhà cung cấp
- "invoice_date": ngày hóa đơn, định dạng YYYY-MM-DD
- "payment_terms": một trong ["Net 30", "Net 60", "2/10 Net 30"], suy luận từ điều khoản ghi trên hóa đơn,
  nếu không thấy ghi rõ thì để null (KHÔNG tự đoán bừa)
- "due_date": ngày đến hạn nếu có ghi rõ, không thì null
- "gross_amount": tổng tiền trước điều chỉnh (số, đơn vị VNĐ, không có dấu phẩy/chấm phân cách)
- "credit_note_amount": số tiền giảm trừ/hàng trả lại nếu hóa đơn có ghi chú, không thì 0

Nếu một trường không xác định được từ text, để null — KHÔNG bịa số liệu.

Văn bản OCR:
---
{ocr_text}
---
JSON:"""


def build_qwen(model_name: str = QWEN_MODEL_NAME):
    """Load Qwen 1 lần. Bản 3B ở fp16 chỉ cần ~6GB VRAM — không cần quant (bitsandbytes/AWQ),
    tránh toàn bộ nhóm lỗi liên quan (tràn RAM lúc quant runtime, package quant ngừng bảo trì)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype="auto",  # "torch_dtype" đã deprecated trong transformers bản mới, dùng "dtype"
        device_map="auto",
    )
    return tokenizer, model


def qwen_structure_invoice(tokenizer, model, ocr_text: str) -> dict:
    """Gọi Qwen với prompt ràng buộc schema, parse JSON output.
    Trả về dict rỗng nếu model trả về JSON không hợp lệ (để bước Clean xử lý tiếp).
    
    Lưu ý: apply_chat_template() có thể trả về tensor thuần HOẶC BatchEncoding
    (object dict-like của transformers, subscriptable qua ["input_ids"] nhưng
    KHÔNG có .shape trực tiếp) tùy phiên bản/tham số -> check bằng hasattr("shape"),
    không dùng isinstance(..., dict) vì BatchEncoding không kế thừa dict thuần."""
    prompt = EXTRACTION_SCHEMA_PROMPT.format(ocr_text=ocr_text[:4000])  # cắt bớt tránh vượt context
    messages = [{"role": "user", "content": prompt}]
    chat_output = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")

    if hasattr(chat_output, "shape"):
        inputs = chat_output.to(model.device)
    else:
        inputs = chat_output["input_ids"].to(model.device)

    output = model.generate(inputs, max_new_tokens=512, do_sample=False)
    raw = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

    # Model đôi khi bọc JSON trong ```json ... ``` dù đã cấm — chỉ lấy phần {...} đầu tiên cho an toàn
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


# =============================================================
# BƯỚC 3 — CLEAN: validate output của Qwen trước khi đưa vào phân tích
# =============================================================

def clean_extracted_record(record: dict, invoice_id: str) -> dict | None:
    """Loại bỏ hoặc gắn cờ record thiếu field bắt buộc — không để lọt dữ liệu rác
    vào pipeline phân tích công nợ (kế thừa nguyên tắc Data Integrity của ap_analysis.py)."""
    required = ["vendor_name", "invoice_date", "gross_amount"]
    if any(record.get(k) in (None, "", 0) for k in required if k != "gross_amount") or record.get("gross_amount") in (None, 0):
        return None  # thiếu field bắt buộc -> đẩy sang hàng chờ soát thủ công, không auto-nhét vào báo cáo

    try:
        invoice_date = pd.to_datetime(record["invoice_date"])
    except (ValueError, TypeError):
        return None

    payment_terms = record.get("payment_terms")
    if payment_terms not in terms_list:
        payment_terms = None  # để trống thay vì đoán, tránh sai lệch DPO/aging về sau

    due_date = pd.to_datetime(record["due_date"]) if record.get("due_date") else None
    if payment_terms and due_date is None:
        term_days = 60 if payment_terms == "Net 60" else 30
        due_date = invoice_date + pd.Timedelta(days=term_days)

    gross = float(record["gross_amount"])
    credit_note = float(record.get("credit_note_amount") or 0)

    return {
        "Invoice_ID": invoice_id,
        "Legal_Invoice_No": record.get("legal_invoice_no"),
        "Invoice_Symbol": record.get("invoice_symbol"),
        "Vendor_Name": record["vendor_name"],
        "Invoice_Date": invoice_date,
        "Payment_Terms": payment_terms,
        "Due_Date": due_date,
        "Gross_Amount": gross,
        "Credit_Note_Amount": credit_note,
        "Net_Amount": gross - credit_note,
        "Payments": [],  # OCR hóa đơn không có thông tin thanh toán — nạp riêng từ sổ quỹ/ngân hàng
        "Date_Anomaly": bool(due_date is not None and invoice_date > due_date),
        "Possible_Duplicate": False,  # đối chiếu trùng lặp thực hiện ở cấp toàn bộ batch, xem bước Export
    }


# =============================================================
# BƯỚC 4 — EXPORT: gộp batch, phát hiện trùng lặp, trả DataFrame cho analyze()
# =============================================================

def extract_invoices_from_folder(folder: str, lang: str = "vi") -> pd.DataFrame:
    """Chạy toàn bộ pipeline trên 1 thư mục ảnh/scan hóa đơn, trả về DataFrame
    có thể truyền thẳng vào analyze() của ap_analysis.py."""
    ocr_engine = build_ocr_engine(lang)
    tokenizer, model = build_qwen()

    records = []
    skipped = []
    for i, path in enumerate(sorted(Path(folder).glob("*.*")), start=1):
        invoice_id = f"OCR-{datetime.now():%Y%m}-{i:03d}"
        text = ocr_extract_text(ocr_engine, str(path))
        raw_record = qwen_structure_invoice(tokenizer, model, text)
        clean = clean_extracted_record(raw_record, invoice_id)
        if clean is None:
            skipped.append(path.name)  # đưa vào hàng chờ soát thủ công, không âm thầm bỏ qua
            continue
        records.append(clean)

    df_out = pd.DataFrame(records)
    if not df_out.empty:
        df_out["Possible_Duplicate"] = df_out.duplicated(
            subset=["Vendor_Name", "Gross_Amount", "Invoice_Date"], keep=False
        )

    if skipped:
        print(f"[!] {len(skipped)} file cần soát thủ công (thiếu field bắt buộc): {skipped}")

    return df_out


if __name__ == "__main__":
    df_ocr = extract_invoices_from_folder("./invoices_scan")
    result = analyze(df_ocr, as_of_date=datetime.now().date().isoformat())
    print(f"Trích xuất {len(df_ocr)} hóa đơn hợp lệ, DPO: {result['dpo_dollar_weighted']:.1f} ngày")


Writing paddleocr_qwen_extraction.py


In [5]:
!cp /kaggle/input/datasets/bangta205/anh-hoa-don/*.* invoices_scan/

In [6]:
# Chạy thử — cần có ảnh trong invoices_scan/ trước (xem ghi chú phía trên)
!python paddleocr_qwen_extraction.py

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Fetching 6 files: 100%|███████████████████████████| 6/6 [00:02<00:00,  2.73it/s]
Download complete: : 6.87MB [00:02, 3.16MB/s]              Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Fet

## File 3 — `invoice_validation.py` (cổng kiểm thử pandera)

Không phụ thuộc GPU, chạy nhanh. Import `analyze` từ File 1 đã ghi ra đĩa ở trên.

In [7]:
!pip install -q pandera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 18.7 MB/s eta 0:00:00


In [8]:
%%writefile invoice_validation.py
"""
Lớp kiểm thử dữ liệu (Data Contract) — chèn giữa 2 file đã có:

    paddleocr_qwen_extraction.py  --(df_ocr thô, có thể bẩn)-->  [FILE NÀY]  --(df sạch)-->  ap_analysis.analyze()

Lý do PHẢI đặt ở vị trí này, không phải nơi khác:
- Validate ngược vào df_clean của ap_analysis.py là vô nghĩa: dữ liệu đó sinh ra bằng code,
  chắc chắn sạch theo định nghĩa -> schema sẽ luôn pass, không chứng minh được gì.
- Điểm duy nhất dữ liệu thực sự "bẩn" (do OCR đọc sai số, Qwen suy luận thiếu, ảnh mờ...)
  là ngay sau bước trích xuất. Đây mới là nơi cần cổng kiểm thử.

6 lỗ hổng đã sửa so với bản gốc:
1. Sai tên cột: schema validate "Total_Amount" nhưng pipeline thực tế trả về Gross_Amount /
   Net_Amount / Credit_Note_Amount -> validate() luôn raise ngay ở dòng đầu, không chạy nổi.
2. Thiếu lazy=True: .validate() mặc định fail-fast (dừng ở lỗi ĐẦU TIÊN) và raise
   pandera.errors.SchemaError (số ít) — nhưng code bắt except SchemaErrors (số nhiều, dạng
   tổng hợp). Hai kiểu exception khác nhau -> except không bắt được gì, chương trình crash.
3. Payment_Terms / Due_Date bị nullable=False cứng: OCR/Qwen có thể không đọc được các
   trường này (đã thiết kế để trả None thay vì đoán bừa ở clean_extracted_record) — schema
   gốc sẽ chặn cứng toàn bộ các hóa đơn hợp lệ nhưng thiếu 1 trường phụ.
4. Không có ràng buộc Invoice_ID duy nhất — trùng lặp (đã nêu ở review trước) không được
   pandera bắt, dồn hết trách nhiệm sang detect thủ công.
5. Không kiểm tra quan hệ Credit_Note_Amount <= Gross_Amount và Net_Amount phải khớp phép trừ
   — dữ liệu điều chỉnh sai vẫn lọt qua.
6. Check due_date >= invoice_date không loại trừ NaT trước khi so sánh -> với Due_Date nullable,
   phép so sánh với NaT sẽ luôn False và làm rớt oan những dòng thiếu Due_Date hợp lệ.
"""

from __future__ import annotations

import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Check, Column, DataFrameSchema

invoice_schema = DataFrameSchema(
    {
        "Invoice_ID": Column(str, nullable=False, unique=True),
        "Vendor_Name": Column(str, nullable=False),
        "Invoice_Date": Column(pa.DateTime, nullable=False),
        "Payment_Terms": Column(
            str, Check.isin(["Net 30", "Net 60", "2/10 Net 30"]), nullable=True
        ),
        "Due_Date": Column(pa.DateTime, nullable=True),
        "Gross_Amount": Column(
            float, Check(lambda s: s > 0, error="Gross_Amount phải > 0"), nullable=False
        ),
        "Credit_Note_Amount": Column(
            float, Check(lambda s: s >= 0, error="Credit_Note_Amount không được âm"), nullable=False
        ),
        "Net_Amount": Column(float, Check(lambda s: s >= 0, error="Net_Amount không được âm"), nullable=False),
    },
    checks=[
        Check(
            lambda df: df["Due_Date"].isna() | (df["Due_Date"] >= df["Invoice_Date"]),
            name="check_due_date_after_invoice_date",
            error="Ngày đến hạn nhỏ hơn ngày phát hành hóa đơn",
        ),
        Check(
            lambda df: df["Credit_Note_Amount"] <= df["Gross_Amount"],
            name="check_credit_note_not_exceed_gross",
            error="Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc",
        ),
        Check(
            lambda df: (df["Gross_Amount"] - df["Credit_Note_Amount"] - df["Net_Amount"]).abs() < 1,
            name="check_net_amount_consistency",
            error="Net_Amount không khớp Gross_Amount - Credit_Note_Amount",
        ),
    ],
    strict=False,  # cho phép các cột nghiệp vụ khác (Payments, Date_Anomaly...) đi kèm, không chặn
)


def validate_extracted_data(df_invoices: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Trả về (df_hop_le, df_loi). Không raise ra ngoài — batch OCR luôn có khả năng lẫn
    dòng lỗi, nhiệm vụ của hàm này là TÁCH chứ không phải chặn đứng cả pipeline."""
    try:
        invoice_schema.validate(df_invoices, lazy=True)
        print("✅ Kiểm thử thành công: toàn bộ dữ liệu sạch.")
        return df_invoices, pd.DataFrame()
    except pa.errors.SchemaErrors as err:
        failure_cases = err.failure_cases
        bad_indices = set(failure_cases["index"].dropna().astype(int))
        df_bad = df_invoices.loc[df_invoices.index.isin(bad_indices)]
        df_good = df_invoices.loc[~df_invoices.index.isin(bad_indices)]
        print(f"❌ {len(df_bad)}/{len(df_invoices)} hóa đơn lỗi — đẩy sang hàng chờ kế toán soát lại:")
        print(failure_cases[["column", "check", "failure_case"]].to_string(index=False))
        return df_good, df_bad


# =============================================================
# DEMO DEPLOYMENT: OCR (giả lập) -> cổng validate -> báo cáo công nợ
# =============================================================
if __name__ == "__main__":
    from ap_analysis import analyze

    # Giả lập output thô của paddleocr_qwen_extraction.py: trộn hóa đơn sạch + các lỗi
    # điển hình thực tế của OCR/LLM (đọc sai số âm, suy luận sai ngày, trùng mã, thiếu field)
    df_ocr_raw = pd.DataFrame(
        [
            dict(Invoice_ID="OCR-001", Vendor_Name="Công ty CP Thép Hòa An",
                 Invoice_Date=pd.Timestamp("2026-02-01"), Payment_Terms="Net 30",
                 Due_Date=pd.Timestamp("2026-03-03"), Gross_Amount=120_000_000.0,
                 Credit_Note_Amount=0.0, Net_Amount=120_000_000.0, Payments=[]),
            dict(Invoice_ID="OCR-002", Vendor_Name="Bao bì Toàn Cầu",
                 Invoice_Date=pd.Timestamp("2026-02-10"), Payment_Terms=None,  # Qwen không đọc được
                 Due_Date=pd.NaT, Gross_Amount=45_000_000.0,
                 Credit_Note_Amount=0.0, Net_Amount=45_000_000.0, Payments=[]),
            dict(Invoice_ID="OCR-003", Vendor_Name="Hóa chất Việt Đức",
                 Invoice_Date=pd.Timestamp("2026-02-15"), Payment_Terms="Net 30",
                 Due_Date=pd.Timestamp("2026-01-01"),  # PaddleOCR đọc nhầm ngày -> due < invoice
                 Gross_Amount=80_000_000.0, Credit_Note_Amount=0.0, Net_Amount=80_000_000.0, Payments=[]),
            dict(Invoice_ID="OCR-001", Vendor_Name="Công ty CP Thép Hòa An",  # mã trùng
                 Invoice_Date=pd.Timestamp("2026-02-01"), Payment_Terms="Net 30",
                 Due_Date=pd.Timestamp("2026-03-03"), Gross_Amount=120_000_000.0,
                 Credit_Note_Amount=0.0, Net_Amount=120_000_000.0, Payments=[]),
            dict(Invoice_ID="OCR-005", Vendor_Name="Điện máy Công nghiệp Phúc Hưng",
                 Invoice_Date=pd.Timestamp("2026-02-20"), Payment_Terms="Net 60",
                 Due_Date=pd.Timestamp("2026-04-21"), Gross_Amount=-15_000_000.0,  # OCR đọc nhầm dấu trừ
                 Credit_Note_Amount=0.0, Net_Amount=-15_000_000.0, Payments=[]),
        ]
    )

    df_good, df_bad = validate_extracted_data(df_ocr_raw)
    print(f"\n-> {len(df_good)} hóa đơn được nạp vào báo cáo, {len(df_bad)} hóa đơn bị giữ lại.\n")

    if not df_good.empty:
        result = analyze(df_good, as_of_date="2026-04-15")
        print(f"DPO (dollar-weighted) trên batch OCR sạch: {result['dpo_dollar_weighted']}")


Writing invoice_validation.py


In [9]:
# Chạy demo — 5 hóa đơn giả lập (1 sạch, 4 lỗi điển hình)
!python invoice_validation.py

❌ 4/5 hóa đơn lỗi — đẩy sang hàng chờ kế toán soát lại:
            column                                          check                   failure_case
        Net_Amount    Ngày đến hạn nhỏ hơn ngày phát hành hóa đơn                     80000000.0
          Payments    Ngày đến hạn nhỏ hơn ngày phát hành hóa đơn                             []
        Net_Amount Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc                    -15000000.0
Credit_Note_Amount Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc                            0.0
      Gross_Amount Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc                    -15000000.0
          Due_Date Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc            2026-04-21 00:00:00
     Payment_Terms Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc                         Net 60
      Invoice_Date Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc            2026-02-20 00:00:00
       Vendor_Name Số tiền giảm trừ lớn hơn tổng tiền hóa đơn gốc Điện 

## File 4 — `streamlit_app.py` (UI, qua tunnel ngrok)

Streamlit chạy như **tiến trình riêng** (`streamlit run`), không chia sẻ bộ nhớ với kernel
notebook — vẫn cần `ap_analysis.py`/`invoice_validation.py` tồn tại trên đĩa (đã ghi ở File 1
& 3) để `import` bình thường, y hệt lúc chạy local.

Kaggle không cho truy cập trực tiếp qua `localhost` như máy cá nhân → cần **ngrok** lấy link
public tạm thời. Đăng ký free tại https://dashboard.ngrok.com/signup, lấy authtoken dán vào
ô `NGROK_AUTH_TOKEN` ở cell "Khởi chạy" bên dưới.

In [10]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 101.9 MB/s eta 0:00:0000:010:01


In [11]:
%%writefile streamlit_app.py
"""
Demo tương tác cho khách hàng/người dùng cuối (không cần biết code).
Chạy: streamlit run streamlit_app.py

Có 2 CHẾ ĐỘ tách biệt rõ ràng — xem giải thích Web vs Local ở cuối file:

- "Demo mẫu"   : dữ liệu giả lập (ap_analysis.generate_invoices) -> AN TOÀN để deploy công khai
                 lên web (Streamlit Cloud / VPS) cho khách xem trước khi ký hợp đồng.
- "Dữ liệu thật": upload hóa đơn thật (ảnh -> OCR/Qwen, hoặc CSV đã trích xuất sẵn) ->
                 BẮT BUỘC chạy local/on-prem, không public lên internet.
"""

import io
from datetime import datetime

import pandas as pd
import plotly.express as px
import streamlit as st

from ap_analysis import generate_invoices, analyze
from invoice_validation import validate_extracted_data

st.set_page_config(page_title="Quản trị Công nợ Nhà cung cấp", layout="wide")
st.title("📊 Dashboard Quản trị Công nợ & Vốn lưu động")

mode = st.sidebar.radio(
    "Chọn chế độ",
    ["🧪 Demo mẫu (dữ liệu giả lập)", "📁 Dữ liệu thật (upload CSV đã trích xuất)"],
)
as_of = st.sidebar.date_input("Chốt sổ tính đến ngày", value=datetime(2026, 4, 15))

# -------------------------------------------------------------
# Chế độ 1: Demo mẫu — luôn chạy được, không đụng dữ liệu thật
# -------------------------------------------------------------
if mode.startswith("🧪"):
    st.info("Đang xem **dữ liệu mẫu** để minh họa, không phải số liệu thật của công ty bạn.")
    n = st.sidebar.slider("Số hóa đơn mẫu", 20, 200, 50)
    df = generate_invoices(n)
    df_valid = df  # dữ liệu sinh sẵn, không cần qua cổng validate

# -------------------------------------------------------------
# Chế độ 2: Dữ liệu thật — bắt buộc qua cổng kiểm thử trước khi hiển thị
# -------------------------------------------------------------
else:
    st.warning(
        "⚠️ Chế độ này xử lý dữ liệu công nợ thật — chỉ nên chạy trên máy nội bộ/on-prem, "
        "KHÔNG dùng bản deploy công khai cho chế độ này (xem ghi chú cuối trang)."
    )
    uploaded = st.file_uploader(
        "Upload file CSV hóa đơn (cột: Invoice_ID, Vendor_Name, Invoice_Date, Payment_Terms, "
        "Due_Date, Gross_Amount, Credit_Note_Amount, Net_Amount)",
        type=["csv"],
    )
    if uploaded is None:
        st.stop()

    raw = pd.read_csv(uploaded, parse_dates=["Invoice_Date", "Due_Date"])
    raw["Payments"] = [[] for _ in range(len(raw))]  # placeholder, nạp riêng từ sổ quỹ nếu có
    df_valid, df_bad = validate_extracted_data(raw)

    if not df_bad.empty:
        with st.expander(f"❌ {len(df_bad)} hóa đơn bị giữ lại — cần kế toán soát tay"):
            st.dataframe(df_bad, use_container_width=True)

    if df_valid.empty:
        st.error("Không có hóa đơn nào qua được kiểm thử. Vui lòng kiểm tra lại file CSV.")
        st.stop()
    st.success(f"✅ {len(df_valid)} hóa đơn hợp lệ, sẵn sàng phân tích.")

# -------------------------------------------------------------
# Phân tích & hiển thị (dùng chung cho cả 2 chế độ)
# -------------------------------------------------------------
result = analyze(df_valid, as_of_date=as_of.isoformat())

col1, col2, col3 = st.columns(3)
col1.metric("DPO (dollar-weighted)", f"{result['dpo_dollar_weighted']:.1f} ngày")
col2.metric("Chiết khấu sớm bị bỏ lỡ", f"{result['missed_discount_total']:,.0f} VNĐ")
col3.metric("Hóa đơn nghi trùng lặp", result['flags']['possible_duplicates'])

st.subheader("Bảng phân bổ tuổi nợ (Aging Schedule)")
aging = result["aging_report"]
if not aging.empty:
    fig_aging = px.bar(aging, x="Aging_Bucket", y="Tong_no_VND", text="So_luong_HD",
                        labels={"Tong_no_VND": "Dư nợ (VNĐ)", "Aging_Bucket": ""})
    st.plotly_chart(fig_aging, use_container_width=True)
else:
    st.write("Không có dư nợ mở tại thời điểm chốt sổ.")

col4, col5 = st.columns(2)
with col4:
    st.subheader("Tập trung rủi ro nhà cung cấp")
    vc = result["vendor_concentration_pct"]
    if len(vc):
        st.plotly_chart(px.pie(values=vc.values, names=vc.index), use_container_width=True)
    else:
        st.write("Không có dư nợ quá hạn.")

with col5:
    st.subheader("Dự báo dòng tiền ra sắp tới")
    forecast = result["cash_outflow_forecast"]
    st.plotly_chart(
        px.bar(x=[f"{h} ngày" for h in forecast], y=list(forecast.values()),
               labels={"x": "", "y": "VNĐ cần chuẩn bị"}),
        use_container_width=True,
    )

with st.expander("Xem dữ liệu chi tiết"):
    st.dataframe(df_valid, use_container_width=True)

st.divider()
st.caption(
    "Demo mẫu dùng dữ liệu giả lập — an toàn để chia sẻ link công khai cho khách xem trước. "
    "Chế độ dữ liệu thật xử lý số liệu công nợ nội bộ — chỉ chạy trên máy/server nội bộ, "
    "không public lên internet."
)


Writing streamlit_app.py


In [12]:
# Khởi chạy Streamlit ngầm + mở tunnel ngrok để lấy link public
import subprocess, time
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "36ywysaWvIQxgAgCrq2mk80vf8q_2t4mM7q16d4sQF8bDsUgv"  # https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

proc = subprocess.Popen([
    "streamlit", "run", "streamlit_app.py",
    "--server.port", "8501", "--server.headless", "true"
])
time.sleep(8)  # đợi Streamlit khởi động xong trước khi mở tunnel

public_url = ngrok.connect(8501)
print(f"App đang chạy tại: {public_url}")
print("Để cell này CHẠY LIÊN TỤC (không interrupt) thì app mới còn sống.")

2026-09-13 20:35:14.469 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.187.61.97:8501

App đang chạy tại: NgrokTunnel: "https://polymorphistic-unshaking-stacee.ngrok-free.dev" -> "http://localhost:8501"
Để cell này CHẠY LIÊN TỤC (không interrupt) thì app mới còn sống.


2026-09-13 20:36:22.898 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-13 20:36:22.959 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-13 20:36:22.983 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-13 20:36:22.985 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-13 20:37:34.774 

---
## Ghi chú vận hành

- **Tắt app:** Interrupt cell khởi chạy (nút ⏹), hoặc gọi `ngrok.kill()` + `proc.terminate()`.
- **Kaggle session timeout:** notebook rảnh quá lâu tự ngắt (giới hạn free tier) — link ngrok
  chết theo, chạy lại cell khởi chạy để lấy link mới.
- **Đổi ảnh hóa đơn:** xóa `invoices_scan/`, upload ảnh mới, chạy lại 3 cell của File 2.
- **Tải file `.py` về máy:** tab "Output" bên phải sau khi chạy, hoặc commit notebook để
  Kaggle tự lưu `/kaggle/working/`.

In [15]:
ngrok.kill()